# CrewAI Research Assistant — Llama 3.2 3B (Q4_K_M) with llama.cpp

This Google Colab notebook demonstrates a single-agent CrewAI research task using **Llama 3.2 3B Instruct**, quantized as **Q4_K_M**, served locally by **llama.cpp**.

**Runtime notes**
- Use a Colab runtime with enough RAM/disk for the GGUF model (roughly a few GB).
- A GPU is recommended but not required. The build cell selects CUDA when a CUDA GPU is available and otherwise builds CPU-only.
- The first setup/build and model download can take several minutes.
- The model is downloaded from Hugging Face; ensure the runtime has internet access.
- No Ollama, ChatOllama, or Qwen is used in this notebook.


## Cell 1

In [ ]:
import sys
print("Python version:", sys.version)
assert (3, 10) <= sys.version_info[:2] < (3, 14), (
    "CrewAI requires Python 3.10–3.13."
)

%pip install -q crewai huggingface_hub requests

## Cell 2

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from importlib.metadata import version

print("CrewAI installed successfully!")
print("CrewAI version:", version("crewai"))

## Cell 3

In [ ]:
import os
import subprocess
import shutil
from pathlib import Path

HAS_CUDA = shutil.which("nvidia-smi") is not None
if HAS_CUDA:
    gpu_check = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=False
    )
    HAS_CUDA = gpu_check.returncode == 0

print("CUDA GPU available:", HAS_CUDA)
print("Build mode:", "CUDA" if HAS_CUDA else "CPU")

## Cell 4

In [ ]:
# Build the official llama.cpp server.
# This may take several minutes in a fresh Colab runtime.
%cd /content
if not Path("/content/llama.cpp").exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git

%cd /content/llama.cpp
!cmake --version

if HAS_CUDA:
    !cmake -S . -B build -DGGML_CUDA=ON -DCMAKE_BUILD_TYPE=Release
else:
    !cmake -S . -B build -DCMAKE_BUILD_TYPE=Release

!cmake --build build --config Release --target llama-server -j 2

server_candidates = [
    Path("/content/llama.cpp/build/bin/llama-server"),
    Path("/content/llama.cpp/build/bin/server"),
]
LLAMA_SERVER = next((p for p in server_candidates if p.exists()), None)
if LLAMA_SERVER is None:
    raise FileNotFoundError("Could not find the built llama-server executable.")
print("llama.cpp server:", LLAMA_SERVER)

## Cell 5

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_REPO = "bartowski/Llama-3.2-3B-Instruct-GGUF"
MODEL_FILE = "Llama-3.2-3B-Instruct-Q4_K_M.gguf"
MODEL_PATH = hf_hub_download(
    repo_id=MODEL_REPO,
    filename=MODEL_FILE,
    local_dir="/content/models/llama-3.2-3b",
)
print("Model downloaded to:", MODEL_PATH)
print("Quantization: Q4_K_M")

## Cell 6

In [ ]:
# Start llama.cpp's OpenAI-compatible HTTP server.
# If a previous run left a server process alive, restart the runtime or stop it first.
import time
import requests

SERVER_PORT = 8000
SERVER_URL = f"http://127.0.0.1:{SERVER_PORT}/v1"

server_args = [
    str(LLAMA_SERVER),
    "-m", MODEL_PATH,
    "--host", "127.0.0.1",
    "--port", str(SERVER_PORT),
    "-c", "4096",
    "-a", "llama-3.2-3b-instruct",
]
if HAS_CUDA:
    server_args += ["-ngl", "99"]
else:
    server_args += ["-ngl", "0"]

server_log = open("/content/llama-server.log", "w")
server_process = subprocess.Popen(
    server_args,
    stdout=server_log,
    stderr=subprocess.STDOUT,
)

for attempt in range(90):
    try:
        health = requests.get(f"{SERVER_URL}/models", timeout=2)
        if health.ok:
            break
    except requests.RequestException:
        pass
    if server_process.poll() is not None:
        raise RuntimeError(
            "llama-server exited during startup. Inspect /content/llama-server.log."
        )
    time.sleep(2)
else:
    raise TimeoutError(
        "llama-server did not become ready. Inspect /content/llama-server.log."
    )

print("llama.cpp server is ready:", SERVER_URL)
print(health.json())

## Cell 7

In [ ]:
# CrewAI connects to llama.cpp through its OpenAI-compatible API.
# The API key is a placeholder; llama.cpp's local server does not require a real key.
llm = LLM(
    model="openai/llama-3.2-3b-instruct",
    base_url=SERVER_URL,
    api_key="sk-no-key-required",
    temperature=0.2,
)

researcher = Agent(
    role="AI Research Assistant",
    goal="Explain artificial intelligence concepts clearly and accurately.",
    backstory=(
        "You are a careful research assistant who explains technical concepts "
        "to beginners and distinguishes established facts from uncertainty."
    ),
    llm=llm,
    verbose=True,
)

print("CrewAI agent configured with llama.cpp + Llama 3.2 3B Q4_K_M.")

## Cell 8

In [ ]:
research_task = Task(
    description=(
        "Explain what a Large Language Model (LLM) is in approximately 150 words. "
        "Include one practical example. Explain the concept in your own words and "
        "avoid claiming that LLMs always produce correct answers."
    ),
    expected_output=(
        "A clear, accurate explanation of LLMs in approximately 150 words, "
        "including one practical example and a brief limitation."
    ),
    agent=researcher,
)

print("Research task created successfully!")

## Cell 9

In [ ]:
crew = Crew(
    agents=[researcher],
    tasks=[research_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew assembled successfully!")

## Cell 10

In [ ]:
result = await crew.kickoff_async()

print("\n========== RESEARCH RESULTS ==========\n")
print(result)

## Cell 11

In [ ]:
# Optional smoke test: query the local OpenAI-compatible endpoint directly.
# This helps separate server/API issues from CrewAI configuration issues.
response = requests.post(
    f"{SERVER_URL}/chat/completions",
    json={
        "model": "llama-3.2-3b-instruct",
        "messages": [
            {"role": "user", "content": "Reply with one short sentence confirming you are running."}
        ],
        "temperature": 0.1,
        "max_tokens": 80,
    },
    timeout=120,
)
response.raise_for_status()
print(response.json()["choices"][0]["message"]["content"])

## Troubleshooting

- **Build fails:** Restart the Colab runtime and rerun setup. If CUDA compilation fails on a particular runtime image, change Cell 4 to a CPU-only build by setting `HAS_CUDA = False`, then rebuild.
- **Model download fails:** Check the repository and filename in Cell 5 and verify internet access.
- **Server does not start:** Inspect `/content/llama-server.log`. GPU memory may be insufficient; try a CPU runtime or reduce context length from `4096` to `2048`.
- **CrewAI cannot connect:** Confirm Cell 6 reports the server as ready and that Cell 7 uses the same `SERVER_URL`.
- **Runtime reset:** Colab deletes the built server and downloaded model when its temporary runtime storage is cleared; rerun the setup cells.

The notebook builds the official `llama.cpp` server and uses its OpenAI-compatible endpoint. The model output is generated by the local GGUF model, not by a hosted Ollama service.
